# Data types and precision

ffsim represents state vectors as one-dimensional Numpy arrays. By default these are double-precision complex arrays (`complex128`). You can also simulate in single precision (`complex64`). This halves the memory used by each state vector and speeds up the gate kernels, at the cost of accuracy.

## Choosing a precision

Pass `dtype` when you create the initial state, then apply gates as usual:

In [1]:
import numpy as np

import ffsim

norb, nelec = 8, (4, 4)
ucj_op = ffsim.random.random_ucj_op_spin_balanced(norb, n_reps=2, seed=1)

vec = ffsim.hartree_fock_state(norb, nelec, dtype=np.complex64)
vec = ffsim.apply_unitary(vec, ucj_op, norb, nelec)
print(vec.dtype)

complex64


[slater_determinant](../api/stubs/ffsim.slater_determinant.rst#ffsim.slater_determinant) and [final_state_vector](../api/stubs/ffsim.qiskit.final_state_vector.rst#ffsim.qiskit.final_state_vector) also accept `dtype`. You can also convert an existing vector with `vec.astype(np.complex64)`.

## How each kind of function handles data types

| Function | Input data type | Computed in | Output data type |
|---|---|---|---|
| Gates (`ffsim.apply_*`), [apply_unitary](../api/stubs/ffsim.apply_unitary.rst#ffsim.apply_unitary), and Trotter simulation | `complex64` or `complex128` | input precision | same as input |
| | `float32` | single precision | `complex64` |
| | `float64` | double precision | `complex128` |
| Linear operators ([linear_operator](../api/stubs/ffsim.linear_operator.rst#ffsim.linear_operator)) | any | double precision | `float64` if both the operator and the vector are real, otherwise `complex128` |
| Reduced density matrices, [spin_square](../api/stubs/ffsim.spin_square.rst#ffsim.spin_square) | any | double precision | double precision |

- **Gate parameters**: orbital rotation matrices, coefficients, and angles are always processed in double precision. The phases and rotation coefficients the gates use are rounded to single precision only when they are applied to a `complex64` vector.
- **Linear operators**: these use PySCF's FCI routines, which only support double precision, so a single-precision vector is converted to double precision before the operator is applied.
- **UCCSD operators**: applying a UCCSD operator goes through a linear operator, so it is also computed in double precision. The result is converted back to the data type of the input vector.
- **Unsupported types**: extended-precision types such as `clongdouble` raise a `TypeError`.

## Accuracy

Single-precision arithmetic has a relative precision of about $10^{-7}$. Each gate adds error of roughly this size to the state vector. For example, one layer of an LUCJ ansatz on 14 spatial orbitals gives a state that differs from the double-precision result by about $10^{-6}$ in the 2-norm. The error grows with circuit depth. The following cell compares the single-precision state computed above with the double-precision result.

In [2]:
vec_double = ffsim.hartree_fock_state(norb, nelec)
vec_double = ffsim.apply_unitary(vec_double, ucj_op, norb, nelec)
print(vec_double.dtype)
print(np.linalg.norm(vec - vec_double))

complex128
5.333363895163611e-07


Use double precision when you need high accuracy, for example when computing energies to chemical accuracy after many gates, or when computing gradients by finite differences.